# PINNs vs FDM for Boundary Value Problems
  
Implementation notebook following the theory review.  
Each section: ELI5 explanation, key ideas, equations, then code.

---

# 0. Motivation

## ELI5

We have a differential equation with boundary conditions (a BVP). There are two
ways to solve it numerically: the classical way (FDM, which the course taught)
and a new machine-learning way (PINNs). We want to know which is better and why.

## Key Ideas

- **FDM** replaces derivatives with algebra on a grid. Convergence is *proven* by theory.
- **PINNs** train a neural network to satisfy the equation. There is *no convergence guarantee*.
- Both produce approximate solutions. We compare accuracy, cost, and failure modes.

## Research Questions

1. How does accuracy scale with resolution ($n$ for FDM vs $N_c$ for PINNs)?
2. Can Richardson extrapolation recover $O(h^4)$ from two $O(h^2)$ solves?
3. Where do PINNs break down relative to FDM?
4. How does FDM conditioning compare to PINN loss landscape pathologies?

In [26]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import time

np.random.seed(42)
torch.manual_seed(42)

plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 6)})

---
# 1. Test Problems

## ELI5

We pick 3 equations where we know the exact answer, so we can measure how
wrong each method is.

## Key Ideas

- All 3 are second-order BVPs on $[0,1]$.
- Prob 1: Poisson equation, analytic condition number (tests conditioning).
- Prob 2: variable coefficient, non-polynomial exact solution (tests spectral bias).
- Prob 3: nonlinear (tests iterative solvers).

## Equations

**Problem 1** -- Poisson:
$$y'' = -\pi^2\sin(\pi x), \quad y(0)=y(1)=0$$
$$y_e(x) = \sin(\pi x)$$

**Problem 2** -- Linear, variable coefficient:
$$\frac{d}{dx}\left[(1+x^2)\frac{dv}{dx}\right] = d(x), \quad v(0)=1,\; v(1)=2+\sin(1)$$
$$v_e(x) = x^2 + \sin(x) + 1$$

where $d(x) = 2 + 6x^2 + 2x\cos(x) - (1+x^2)\sin(x)$.

**Problem 3** -- Nonlinear:
$$v'' = 3v + x^2 + 10v^3, \quad v(0)=0,\; v(1)=0.1$$

No closed form. Reference solution via Newton shooting at tol $10^{-12}$.

## Why These Specifically

- **Prob 1** gives the canonical tridiagonal matrix $A_h = \text{tridiag}(1,-2,1)/h^2$
  whose eigenvalues $\lambda_k = 2 - 2\cos(k\pi/n)$ and condition number
  $\kappa(A_h) \approx 4n^2/\pi^2$ are known analytically. Cleanest conditioning benchmark.
- **Prob 2** uses $\sin(x)$ instead of a polynomial because a neural network fits
  polynomials trivially, making the comparison meaningless. The $\sin(x)$ component
  exercises **spectral bias** directly.
- **Prob 3** at $\beta = 0.1 \ll \beta_c \approx 3.95$ ensures existence and uniqueness.
  Tests all 4 nonlinear solvers.

In [27]:
# --- Exact solutions and problem definitions ---

BETA = 0.1  # Problem 3 right BC


def exact2(x):
    """Problem 2 exact solution."""
    return x**2 + np.sin(x) + 1


def exact1(x):
    """Problem 1 exact solution."""
    return np.sin(np.pi * x)


def rhs2(x):
    """Problem 2 RHS: d(x)."""
    return 2 + 6 * x**2 + 2 * x * np.cos(x) - (1 + x**2) * np.sin(x)


def rhs1(x):
    """Problem 1 RHS."""
    return -np.pi**2 * np.sin(np.pi * x)


def a_coeff(x):
    """Problem 2 variable coefficient a(x) = 1 + x^2."""
    return 1 + x**2


# Problem boundary conditions: (left_bc, right_bc)
BC2 = (1.0, 2.0 + np.sin(1.0))
BC1 = (0.0, 0.0)
BC3 = (0.0, BETA)

In [28]:
# Quick plot of exact solutions
x_plot = np.linspace(0, 1, 500)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(x_plot, exact1(x_plot))
axes[0].set_title(r'Problem 1: $\sin(\pi x)$')
axes[1].plot(x_plot, exact2(x_plot))
axes[1].set_title(r'Problem 2: $x^2 + \sin(x) + 1$')
axes[2].set_title('Problem 3: no closed form')
axes[2].text(0.5, 0.5, '(reference solution\ncomputed below)',
             ha='center', va='center', transform=axes[2].transAxes, color='gray')
for ax in axes:
    ax.set_xlabel('$x$')
    ax.set_ylabel('$v(x)$')
plt.tight_layout()
plt.show()

/tmp/ipykernel_1854851/696382712.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
# 2. FDM -- Implementation

## ELI5

Replace the continuous domain with a grid of points. Replace derivatives with
differences between neighbouring grid values. This turns the differential equation
into a system of linear equations that we can solve directly.

## Key Ideas

**Discretisation:**
- Put $n$ equally-spaced interior points on $[0,1]$, spacing $h = 1/(n+1)$.
- Approximate $y''$ at each grid point using its two neighbours.
- This gives a **tridiagonal** linear system $A_h U = F$.
- Solve in $O(n)$ time (Thomas algorithm / sparse solver). Very fast.

**What we implement:**
- **Centered difference** for $y''$ (Problem 1): uses the 3-point stencil (Notes, eq. 6.10).
- **Symmetric differencing** for $(a(x)y')'$ (Problem 2): handles the variable
  coefficient by evaluating $a(x)$ at half-grid points $x_{i \pm 1/2}$ (Slides p.12, A3 Q1b).
- **Boundary corrections** for Problem 2: the non-zero BCs modify the first
  and last rows of the RHS vector.

**Matrix properties (Notes, Section 5.1):**
- Problem 1 produces $A_h = \text{tridiag}(1,-2,1)/h^2$, which is **negative definite**
  because $-A_h = \text{tridiag}(-1,2,-1)/h^2$ satisfies the conditions of Theorem 17
  (symmetric, strictly diagonally dominant, positive diagonal).
- Invertibility follows from Theorem 16 (positive/negative definite $\Rightarrow$ non-singular).

## Equations

**Centered difference** for $y''$ (Notes, eq. 6.10):
$$y''(x_i) = \frac{y_{i-1} - 2y_i + y_{i+1}}{h^2} + O(h^2)$$

Derived by expanding $y(x_i \pm h)$ in Taylor series to 4th order and summing.
The $O(h)$ terms cancel, leaving $O(h^2)$ truncation error.

**Symmetric differencing** for $(a(x)y')'$ (Problem 2, Slides p.12):
$$\frac{1}{h^2}\left[a_{i-1/2}\,y_{i-1} - (a_{i+1/2}+a_{i-1/2})\,y_i + a_{i+1/2}\,y_{i+1}\right]$$

where $a_{i\pm 1/2} = a(x_i \pm h/2) = 1 + (x_i \pm h/2)^2$. Also second-order accurate.

Both formulas produce a **tridiagonal matrix** $A_h$:
- Problem 1: $A_h = \text{tridiag}(1, -2, 1)/h^2$ (constant diagonals)
- Problem 2: $A_h$ has variable entries from $a_{i\pm 1/2}$

**Note on conservative form:** The FDM discretises $(a u')'$ *directly* via symmetric
differencing, preserving the conservative (divergence) form. The PINN implementation
below instead expands via the product rule: $(au')' = a'u' + au''$. Both are valid,
but the FDM approach is numerically preferable for variable-coefficient problems.

In [30]:
def fdm_problem2(n, x_eval=None):
    """
    FDM for Problem 2: d/dx[(1+x^2) dv/dx] = d(x).
    Symmetric differencing for (a(x)y')'.
    """
    h = 1.0 / (n + 1)
    x = np.linspace(0, 1, n + 2)
    xi = x[1:-1]

    # a(x) at half-points
    a_plus = a_coeff(xi + h / 2)   # a_{i+1/2}
    a_minus = a_coeff(xi - h / 2)  # a_{i-1/2}

    # Tridiagonal entries
    main_diag = -(a_plus + a_minus) / h**2
    upper = a_plus[:-1] / h**2
    lower = a_minus[1:] / h**2
    A = sp.diags([lower, main_diag, upper], [-1, 0, 1], format='csc')

    # RHS with boundary corrections
    F = rhs2(xi).copy()
    F[0] -= a_minus[0] * BC2[0] / h**2   # v(0) = 1
    F[-1] -= a_plus[-1] * BC2[1] / h**2   # v(1) = 2 + sin(1)

    U = spla.spsolve(A, F)

    u_full = np.concatenate([[BC2[0]], U, [BC2[1]]])

    if x_eval is not None:
        u_full = np.interp(x_eval, x, u_full)
        e = np.max(np.abs(u_full - exact2(x_eval)))
        return u_full, e

    e = np.max(np.abs(U - exact2(xi)))
    return u_full, x, e

In [29]:
def fdm_problem1(n, x_eval=None):
    """
    FDM for Problem 1: y'' = -pi^2 sin(pi x), y(0)=y(1)=0.
    Centered difference -> tridiagonal system.
    """
    h = 1.0 / (n + 1)
    x = np.linspace(0, 1, n + 2)  # includes boundaries
    xi = x[1:-1]                   # interior points

    # A_h = tridiag(1, -2, 1) / h^2
    main_diag = -2.0 * np.ones(n) / h**2
    off_diag = 1.0 * np.ones(n - 1) / h**2
    A = sp.diags([off_diag, main_diag, off_diag], [-1, 0, 1], format='csc')

    # RHS (BCs are zero so no correction needed)
    F = rhs1(xi)

    # Solve
    U = spla.spsolve(A, F)

    # Full solution including boundaries
    u_full = np.concatenate([[BC1[0]], U, [BC1[1]]])

    if x_eval is not None:
        u_full = np.interp(x_eval, x, u_full)
        e = np.max(np.abs(u_full - exact1(x_eval)))
        return u_full, e

    e = np.max(np.abs(U - exact1(xi)))
    return u_full, x, e

In [31]:
# Sanity check: solve both problems at n=63 and plot
for label, solver, exact_fn in [('Problem 1', fdm_problem1, exact1),
                                 ('Problem 2', fdm_problem2, exact2)]:
    u, x, e = solver(63)
    plt.figure(figsize=(8, 4))
    plt.plot(x, exact_fn(x), 'k-', label='Exact')
    plt.plot(x, u, 'ro--', markersize=3, label=f'FDM n=63')
    plt.title(f'{label}: FDM vs Exact (error = {e:.2e})')
    plt.legend()
    plt.xlabel('$x$')
    plt.ylabel('$v(x)$')
    plt.tight_layout()
    plt.show()

/tmp/ipykernel_1854851/2375305049.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Convergence Study

### Key Idea

The **BVP Convergence Theorem** (Slides p.7-8; *not* the Lax equivalence theorem
from Notes Theorem 25, which applies to time-marching PDE schemes) guarantees:
$$\|E^h\|_\infty \le h^2 \|A_h^{-1}\|_\infty \|\tau\|_\infty = O(h^2)$$

where $E^h$ is the error vector and $\tau$ is the truncation error.
The bound $\|A_h^{-1}\|_2 = O(h^{-2})$ follows from the eigenvalue formula
$\lambda_k = 2 - 2\cos(k\pi/(n+1))$ (Notes, Section 5.4).

We verify this by computing the **effective convergence rate** (A3 Q1e):
$$r_h = \frac{\ln(e_h / e_{h/2})}{\ln 2} \to 2$$

If $r_h \to 2$, the method is second-order as predicted.

## Richardson Extrapolation

### ELI5

If we know the error is roughly $Ch^2$, we can run the solver at two different
grid sizes, and cleverly combine the results to cancel the $h^2$ term. This
gives $O(h^4)$ accuracy from two $O(h^2)$ solves -- no new stencil, no new theory.

### Formula

$$u_{\text{rich}} = \frac{4\,u_{h/2} - u_h}{3}, \quad \text{error} = O(h^4)$$

We verify $r_{\text{rich}} \to 4$.

In [32]:
def convergence_study(solver, exact_fn, label):
    """
    Run FDM at doubling n values.
    Compute standard and Richardson errors + convergence rates.
    """
    ns = [7, 15, 31, 63, 127, 255, 511, 1023]
    x_test = np.linspace(0, 1, 1002)[1:-1]  # 1000 interior test points

    errors = []
    for n in ns:
        _, e = solver(n, x_eval=x_test)
        errors.append(e)

    # Convergence rates
    rates = [None]
    for i in range(1, len(errors)):
        rates.append(np.log(errors[i - 1] / errors[i]) / np.log(2))

    # Richardson extrapolation (consecutive pairs)
    rich_errors = [None]
    rich_rates = [None, None]
    for i in range(1, len(ns)):
        u_h, _ = solver(ns[i - 1], x_eval=x_test)
        u_h2, _ = solver(ns[i], x_eval=x_test)
        u_rich = (4 * u_h2 - u_h) / 3
        e_rich = np.max(np.abs(u_rich - exact_fn(x_test)))
        rich_errors.append(e_rich)

    for i in range(2, len(rich_errors)):
        rich_rates.append(np.log(rich_errors[i - 1] / rich_errors[i]) / np.log(2))

    # Print table
    print(f'\n{label}')
    print(f'{"n":>6} {"h":>10} {"e_h":>12} {"r_h":>8} {"e_rich":>12} {"r_rich":>8}')
    print('-' * 62)
    for i, n in enumerate(ns):
        h = 1.0 / (n + 1)
        r = f'{rates[i]:.2f}' if rates[i] is not None else '--'
        er = f'{rich_errors[i]:.2e}' if rich_errors[i] is not None else '--'
        rr = f'{rich_rates[i]:.2f}' if i < len(rich_rates) and rich_rates[i] is not None else '--'
        print(f'{n:>6} {h:>10.6f} {errors[i]:>12.2e} {r:>8} {er:>12} {rr:>8}')

    return ns, errors, rich_errors


ns1, e1, r1 = convergence_study(fdm_problem1, exact1, 'Problem 1')
ns2, e2, r2 = convergence_study(fdm_problem2, exact2, 'Problem 2')


Problem 1
     n          h          e_h      r_h       e_rich   r_rich
--------------------------------------------------------------
     7   0.125000     3.91e-03       --           --       --
    15   0.062500     9.81e-04     2.00     1.22e-03       --
    31   0.031250     2.45e-04     2.00     3.09e-04     1.98
    63   0.015625     6.14e-05     2.00     7.79e-05     1.99
   127   0.007812     1.53e-05     2.00     1.92e-05     2.02
   255   0.003906     3.84e-06     2.00     4.92e-06     1.97
   511   0.001953     9.59e-07     2.00     1.16e-06     2.08
  1023   0.000977     2.40e-07     2.00     3.03e-07     1.94

Problem 2
     n          h          e_h      r_h       e_rich   r_rich
--------------------------------------------------------------
     7   0.125000     1.26e-02       --           --       --
    15   0.062500     3.14e-03     2.01     6.31e-03       --
    31   0.031250     7.79e-04     2.01     1.56e-03     2.02
    63   0.015625     1.92e-04     2.02     3.

In [33]:
# Convergence plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, ns, errors, rich_errors, title in [
    (axes[0], ns1, e1, r1, 'Problem 1'),
    (axes[1], ns2, e2, r2, 'Problem 2'),
]:
    hs = [1.0 / (n + 1) for n in ns]
    ax.loglog(ns, errors, 'bo-', label='FDM $O(h^2)$')
    # Richardson (skip first None)
    rich_ns = [ns[i] for i in range(1, len(ns))]
    rich_es = [rich_errors[i] for i in range(1, len(rich_errors))]
    ax.loglog(rich_ns, rich_es, 'rs-', label='Richardson $O(h^4)$')
    # Reference slopes
    ref_n = np.array([ns[0], ns[-1]])
    ax.loglog(ref_n, errors[0] * (ref_n / ns[0])**(-2), 'b--', alpha=0.3, label='slope $-2$')
    ax.loglog(ref_n, rich_es[0] * (ref_n / rich_ns[0])**(-4), 'r--', alpha=0.3, label='slope $-4$')
    ax.set_xlabel('$n$')
    ax.set_ylabel(r'$L^\infty$ error')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

/tmp/ipykernel_1854851/851901622.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Condition Number (Problem 1)

### ELI5

The condition number $\kappa(A_h)$ measures how much errors in the input get
amplified in the output (Notes, Section 5.4, eq. 5.2):
$$\frac{\|\delta X\|}{\|X\|} \le \kappa(A) \cdot \frac{\|\delta b\|}{\|b\|}$$

For the Poisson matrix, $\kappa$ grows as $n^2$. Eventually,
when $\kappa \cdot \epsilon_{\text{mach}} \approx 1$, we lose all significant digits
and the answer is meaningless.

### Key Ideas

The eigenvalues of the *unscaled* tridiagonal matrix $\text{tridiag}(-1,2,-1)$ are
(Notes, Section 5.4, derived via characteristic equation $\rho^2 - (2-\lambda)\rho + 1 = 0$):
$$\lambda_k = 2 - 2\cos\!\left(\frac{k\pi}{n+1}\right), \quad k = 1, \ldots, n$$

The $1/h^2$ scaling of $A_h$ multiplies every eigenvalue by the same factor,
so it **cancels in the ratio** $\lambda_{\max}/\lambda_{\min}$. The condition number
is the same whether we use the scaled or unscaled matrix:
$$\kappa_2(A_h) = \frac{|\lambda_{\max}|}{|\lambda_{\min}|} = \frac{2 - 2\cos(n\pi/(n+1))}{2 - 2\cos(\pi/(n+1))} \approx \frac{4(n+1)^2}{\pi^2}$$

This is the **analytically characterised failure mode** of FDM:
$\kappa \approx 10^{15}$ when $n \approx 10^7$ (Notes, Section 5.4, p.102).
Compare to PINNs, where the failure mode is empirical and unpredictable.

In [34]:
# Condition number study for Problem 1
ns_cond = [7, 15, 31, 63, 127, 255, 511, 1023, 2047]
kappas = []

for n in ns_cond:
    h = 1.0 / (n + 1)
    main_diag = -2.0 * np.ones(n) / h**2
    off_diag = 1.0 * np.ones(n - 1) / h**2
    A = sp.diags([off_diag, main_diag, off_diag], [-1, 0, 1], format='csc')
    # For small n, compute exactly; for large n, use analytic formula
    if n <= 1023:
        A_dense = A.toarray()
        kappas.append(np.linalg.cond(A_dense))
    else:
        kappas.append(4 * (n + 1)**2 / np.pi**2)

# Analytic reference
ns_ref = np.array(ns_cond, dtype=float)
kappa_analytic = 4 * (ns_ref + 1)**2 / np.pi**2

plt.figure(figsize=(8, 5))
plt.loglog(ns_cond, kappas, 'bo-', label=r'Computed $\kappa(A_h)$')
plt.loglog(ns_ref, kappa_analytic, 'r--', label=r'$4n^2/\pi^2$')
plt.axhline(1 / np.finfo(float).eps, color='gray', ls=':', label=r'$1/\epsilon_{\mathrm{mach}}$')
plt.xlabel('$n$')
plt.ylabel('$\\kappa(A_h)$')
plt.title('Problem 1: Condition Number')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

/tmp/ipykernel_1854851/1009688872.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [35]:
# Backward residual check (Notes, Section 5.4, eq. 5.2)
# Verify: ||A_h Y_h - F_h|| ~ eps_mach * kappa(A_h) * ||F_h||
print('\nBackward Residual Check (Problem 1)')
print(f'{"n":>6} {"kappa":>12} {"||r||/||F||":>14} {"kappa*eps":>14}')
print('-' * 50)
eps = np.finfo(float).eps
for n in [31, 63, 127, 255, 511, 1023]:
    h = 1.0 / (n + 1)
    xi = np.linspace(h, 1 - h, n)
    main_diag = -2.0 * np.ones(n) / h**2
    off_diag = 1.0 * np.ones(n - 1) / h**2
    A = sp.diags([off_diag, main_diag, off_diag], [-1, 0, 1], format='csc')
    F = rhs1(xi)
    U = spla.spsolve(A, F)
    residual = A @ U - F
    rel_res = np.linalg.norm(residual) / np.linalg.norm(F)
    kappa = 4 * (n + 1)**2 / np.pi**2
    print(f'{n:>6} {kappa:>12.2e} {rel_res:>14.2e} {kappa * eps:>14.2e}')


Backward Residual Check (Problem 2)
     n        kappa    ||r||/||F||      kappa*eps
--------------------------------------------------
    31     4.15e+02       1.28e-14       9.22e-14
    63     1.66e+03       5.25e-14       3.69e-13
   127     6.64e+03       2.42e-13       1.47e-12
   255     2.66e+04       7.68e-13       5.90e-12
   511     1.06e+05       3.17e-12       2.36e-11
  1023     4.25e+05       1.23e-11       9.44e-11


---
# 3. Nonlinear Solvers (Problem 3)

## ELI5

Problem 3 is nonlinear, so we can't just build a matrix and solve $Ax = b$.
We need **iterative methods** that guess an answer, check how wrong it is,
and improve. We implement 4 different solvers.

## Shooting Methods -- Convert BVP to Root-Finding

**The idea:**
- Guess the initial slope $\alpha = v'(0)$.
- Solve the resulting IVP forward from $x=0$ to $x=1$ using `solve_ivp`
  (RK45/Dormand-Prince, the scipy equivalent of MATLAB's `ode45` -- Notes, Section 4.6).
- Check if we hit the right boundary: does $v(1) = \beta$?
- If not, adjust $\alpha$ and repeat.

This reduces the BVP to finding the root of:
$$\Phi(\alpha) = v(1; \alpha) - \beta = 0$$

**Bisection:** Bracket the root and halve the interval. Simple but slow.
Linear convergence (Slides p.15-16).

**Newton shooting:** Use the derivative $\Phi'(\alpha)$ to update (Slides p.15):
$$\alpha_{k+1} = \alpha_k - \frac{\Phi(\alpha_k)}{\Phi'(\alpha_k)}$$

To get $\Phi'(\alpha)$, solve a **sensitivity IVP** alongside the main IVP:
$$z_1' = z_2, \quad z_2' = (3 + 30v^2)z_1, \quad z_1(0)=0,\; z_2(0)=1$$
$$\Phi'(\alpha) = z_1(1)$$

Quadratic convergence (Slides p.15-16; Golub Ch5).
Much faster -- typically converges in 5-10 iterations.

In [36]:
def shooting_rhs(t, y):
    """RHS for the augmented 4D system: [v, v', z1, z2]."""
    v, vp, z1, z2 = y
    dvdt = vp
    dvpdt = 3 * v + t**2 + 10 * v**3
    dz1dt = z2
    dz2dt = (3 + 30 * v**2) * z1
    return [dvdt, dvpdt, dz1dt, dz2dt]


def shoot(alpha):
    """Solve IVP with v'(0) = alpha. Returns Phi(alpha) and Phi'(alpha)."""
    y0 = [0, alpha, 0, 1]  # v(0)=0, v'(0)=alpha, z1(0)=0, z2(0)=1
    sol = solve_ivp(shooting_rhs, [0, 1], y0, rtol=1e-13, atol=1e-13)
    v1 = sol.y[0, -1]
    z1_1 = sol.y[2, -1]
    return v1 - BETA, z1_1


def newton_shooting(alpha0, tol=1e-12, max_iter=50):
    """Newton shooting. Returns converged alpha and iteration history."""
    alpha = alpha0
    history = []
    for k in range(max_iter):
        phi, dphi = shoot(alpha)
        history.append(abs(phi))
        if abs(phi) < tol:
            break
        alpha = alpha - phi / dphi
    return alpha, history


def bisection_shooting(a, b, tol=1e-12, max_iter=100):
    """Bisection shooting. Returns converged alpha and iteration history."""
    history = []
    for k in range(max_iter):
        mid = (a + b) / 2
        phi_mid, _ = shoot(mid)
        history.append(abs(phi_mid))
        if abs(phi_mid) < tol or (b - a) / 2 < tol:
            break
        phi_a, _ = shoot(a)
        if phi_a * phi_mid < 0:
            b = mid
        else:
            a = mid
    return mid, history

In [37]:
# Newton shooting -- establish reference solution
alpha_newton, hist_newton = newton_shooting(0.5)
print(f'Newton shooting converged: alpha = {alpha_newton:.15f}')
print(f'Iterations: {len(hist_newton)}')

# Bisection shooting
alpha_bisect, hist_bisect = bisection_shooting(-1.0, 2.0)
print(f'\nBisection converged: alpha = {alpha_bisect:.15f}')
print(f'Iterations: {len(hist_bisect)}')

Newton shooting converged: alpha = 0.004943440649204
Iterations: 5

Bisection converged: alpha = 0.004943440648731
Iterations: 41


In [38]:
# Generate reference solution for Problem 3 (dense, for later comparison)
def get_reference_solution(alpha, n_points=10000):
    """Solve IVP with converged alpha, return dense solution."""
    y0 = [0, alpha, 0, 1]
    t_eval = np.linspace(0, 1, n_points)
    sol = solve_ivp(shooting_rhs, [0, 1], y0, t_eval=t_eval,
                    rtol=1e-13, atol=1e-13)
    return sol.t, sol.y[0]

x_ref3, v_ref3 = get_reference_solution(alpha_newton)


def exact3_interp(x):
    """Interpolate Problem 3 reference solution."""
    return np.interp(x, x_ref3, v_ref3)


plt.figure(figsize=(8, 4))
plt.plot(x_ref3, v_ref3, 'k-')
plt.title(f'Problem 3 reference solution ($\\beta = {BETA}$)')
plt.xlabel('$x$')
plt.ylabel('$v(x)$')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

/tmp/ipykernel_1854851/1012441770.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## FD Iterative Methods -- Stay on the Grid

Instead of converting to an IVP, we can iterate directly on the FD grid.

**Picard (fixed-point iteration):**
- Freeze the nonlinear terms at the previous iterate: $A_h V_{k+1} = H(V_k)$
- Each step solves a *linear* tridiagonal system (fast).
- Convergence requires the map $G(V) = A_h^{-1}H(V)$ to be a contraction,
  i.e., $\|DG(V)\| < 1$ near the solution. This is **not guaranteed** in general
  (Slides p.17).

**Newton FD** (Slides p.18-19):
- Linearise the full nonlinear system $F(V) = A_h V - H(V) = 0$ using the Jacobian:
$$J(V_k)\,\Delta V_k = -F(V_k), \quad V_{k+1} = V_k + \Delta V_k$$
- $J$ is **tridiagonal**: main diagonal $2/h^2 - (3 + 30v_i^2)$, off-diagonals $-1/h^2$.
- Quadratic convergence (Slides p.18-19; Golub Ch5), like Newton shooting. Typically 5-10 iterations.

Both methods below track the same convergence metric: $\|V_{k+1} - V_k\| / \sqrt{n}$.

In [39]:
def picard_fd(n, tol=1e-12, max_iter=500, V0=None):
    """Picard fixed-point iteration for Problem 3."""
    h = 1.0 / (n + 1)
    xi = np.linspace(h, 1 - h, n)

    # A_h = tridiag(-1, 2, -1) / h^2
    main = 2.0 * np.ones(n) / h**2
    off = -1.0 * np.ones(n - 1) / h**2
    A = sp.diags([off, main, off], [-1, 0, 1], format='csc')

    V = V0.copy() if V0 is not None else np.zeros(n)
    history = []

    for k in range(max_iter):
        rhs = -(3 * V + xi**2 + 10 * V**3)
        # BC corrections
        rhs[0] += BC3[0] / h**2   # v(0) = 0
        rhs[-1] += BC3[1] / h**2  # v(1) = beta

        V_new = spla.spsolve(A, rhs)
        diff = np.linalg.norm(V_new - V) / np.sqrt(n)
        history.append(diff)
        V = V_new
        if diff < tol:
            break

    x_full = np.linspace(0, 1, n + 2)
    u_full = np.concatenate([[BC3[0]], V, [BC3[1]]])
    return u_full, x_full, history


def newton_fd(n, tol=1e-12, max_iter=50, V0=None):
    """Newton FD iteration for Problem 3."""
    h = 1.0 / (n + 1)
    xi = np.linspace(h, 1 - h, n)

    V = V0.copy() if V0 is not None else np.zeros(n)
    history = []

    for k in range(max_iter):
        # F(V) = A_h V - (3V + x^2 + 10V^3)
        AV = np.zeros(n)
        AV[0] = (2 * V[0] - V[1]) / h**2 - BC3[0] / h**2
        AV[-1] = (-V[-2] + 2 * V[-1]) / h**2 - BC3[1] / h**2
        for i in range(1, n - 1):
            AV[i] = (-V[i - 1] + 2 * V[i] - V[i + 1]) / h**2

        FV = AV + (3 * V + xi**2 + 10 * V**3)

        # Jacobian: J = A_h - diag(3 + 30*v_i^2)
        jac_main = 2.0 / h**2 + (3 + 30 * V**2)
        jac_off = -1.0 * np.ones(n - 1) / h**2
        J = sp.diags([jac_off, jac_main, jac_off], [-1, 0, 1], format='csc')

        dV = spla.spsolve(J, -FV)
        V = V + dV

        diff = np.linalg.norm(dV) / np.sqrt(n)
        history.append(diff)
        if diff < tol:
            break

    x_full = np.linspace(0, 1, n + 2)
    u_full = np.concatenate([[BC3[0]], V, [BC3[1]]])
    return u_full, x_full, history

In [40]:
# Run both FD solvers
n3 = 127
h3 = 1.0 / (n3 + 1)
xi3 = np.linspace(h3, 1 - h3, n3)
V0_3 = exact3_interp(xi3)
u_picard, x_picard, hist_picard = picard_fd(n3, V0=V0_3)
u_newton_fd, x_newton_fd, hist_newton_fd = newton_fd(n3, V0=V0_3)

print(f'Picard: {len(hist_picard)} iterations')
print(f'Newton FD: {len(hist_newton_fd)} iterations')

# Plot solutions vs reference
plt.figure(figsize=(8, 4))
plt.plot(x_ref3, v_ref3, 'k-', label='Reference (Newton shooting)')
plt.plot(x_picard, u_picard, 'b--', label='Picard FD')
plt.plot(x_newton_fd, u_newton_fd, 'r:', label='Newton FD')
plt.title('Problem 3: Nonlinear Solvers')
plt.legend()
plt.xlabel('$x$')
plt.ylabel('$v(x)$')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Picard: 25 iterations
Newton FD: 5 iterations


/tmp/ipykernel_1854851/2772841146.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [41]:
# Convergence comparison: all 4 nonlinear solvers
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Shooting methods
axes[0].semilogy(hist_newton, 'ro-', label='Newton shooting')
axes[0].semilogy(hist_bisect, 'bs-', label='Bisection')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel(r'$|\Phi(\alpha)|$')
axes[0].set_title('Shooting Methods')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# FD iterative methods
axes[1].semilogy(hist_picard, 'bs-', label='Picard FD')
axes[1].semilogy(hist_newton_fd, 'ro-', label='Newton FD')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Residual')
axes[1].set_title('FD Iterative Methods')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

/tmp/ipykernel_1854851/3141402022.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [42]:
class PINN(nn.Module):
    """Physics-Informed Neural Network for 1D BVPs."""

    def __init__(self, hidden_layers=4, neurons=32):
        super().__init__()
        layers = []
        layers.append(nn.Linear(1, neurons))
        layers.append(nn.Tanh())
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(neurons, neurons))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(neurons, 1))
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        """Raw network output."""
        return self.net(x)


def hard_bc(model, x, g0, g1):
    """
    Hard boundary enforcement:
    u_hat(x) = g0*(1-x) + g1*x + x*(1-x)*NN(x)
    """
    return g0 * (1 - x) + g1 * x + x * (1 - x) * model(x)


def compute_derivatives(u, x):
    """
    Compute du/dx and d2u/dx2 via autograd.
    create_graph=True is mandatory for second derivative.
    """
    du = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    d2u = torch.autograd.grad(du, x, torch.ones_like(du), create_graph=True)[0]
    return du, d2u


print(f'PINN parameters: {sum(p.numel() for p in PINN().parameters())}')

PINN parameters: 3265


In [43]:
def pinn_residual(model, x, problem_id):
    """
    Compute PDE residual for each problem.
    Returns residual such that residual = 0 means the PDE is satisfied.
    """
    x = x.requires_grad_(True)

    if problem_id == 1:
        u = hard_bc(model, x, BC1[0], BC1[1])
        _, d2u = compute_derivatives(u, x)
        rhs_val = -np.pi**2 * torch.sin(np.pi * x)
        return d2u - rhs_val

    elif problem_id == 2:
        u = hard_bc(model, x, BC2[0], BC2[1])
        du, d2u = compute_derivatives(u, x)
        # Product rule expansion: d/dx[(1+x^2) du/dx] = (2x)(du/dx) + (1+x^2)(d2u/dx2)
        # Note: FDM uses symmetric differencing (conservative form); PINN uses expanded form
        a = 1 + x**2
        da = 2 * x
        lhs = da * du + a * d2u
        rhs_val = 2 + 6 * x**2 + 2 * x * torch.cos(x) - (1 + x**2) * torch.sin(x)
        return lhs - rhs_val

    elif problem_id == 3:
        u = hard_bc(model, x, BC3[0], BC3[1])
        _, d2u = compute_derivatives(u, x)
        rhs_val = 3 * u + x**2 + 10 * u**3
        return d2u - rhs_val

In [44]:
def train_pinn(problem_id, N_c, adam_epochs=10000, seed=42):
    """
    Train a PINN for the given problem.
    Phase 1: Adam. Phase 2: L-BFGS.
    Returns model, loss history, and wall-clock time.
    """
    torch.manual_seed(seed)
    model = PINN()

    # Collocation points
    x_train = torch.linspace(0, 1, N_c + 2)[1:-1].unsqueeze(1).requires_grad_(True)  # interior

    loss_history = []
    t_start = time.time()

    # --- Phase 1: Adam ---
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, adam_epochs)

    for epoch in range(adam_epochs):
        optimizer.zero_grad()
        residual = pinn_residual(model, x_train, problem_id)
        loss = torch.mean(residual**2)
        loss.backward()
        optimizer.step()
        scheduler.step()
        if epoch % 500 == 0 or epoch == adam_epochs - 1:
            loss_history.append((epoch, loss.item()))

    # --- Phase 2: L-BFGS ---
    lbfgs = torch.optim.LBFGS(model.parameters(), max_iter=500,
                               line_search_fn='strong_wolfe')
    lbfgs_step = [0]

    def closure():
        lbfgs.zero_grad()
        residual = pinn_residual(model, x_train, problem_id)
        loss = torch.mean(residual**2)
        loss.backward()
        lbfgs_step[0] += 1
        if lbfgs_step[0] % 50 == 0:
            loss_history.append((adam_epochs + lbfgs_step[0], loss.item()))
        return loss

    lbfgs.step(closure)

    # Final loss
    residual = pinn_residual(model, x_train, problem_id)
    final_loss = torch.mean(residual**2).item()
    loss_history.append((adam_epochs + lbfgs_step[0], final_loss))

    wall_time = time.time() - t_start

    return model, loss_history, wall_time


def evaluate_pinn(model, problem_id, n_test=1000):
    """
    Evaluate PINN L-inf error against exact/reference solution.
    """
    x_test = torch.linspace(0, 1, n_test + 2)[1:-1].unsqueeze(1)
    x_np = x_test.detach().numpy().flatten()

    with torch.no_grad():
        if problem_id == 1:
            u_pred = hard_bc(model, x_test, BC1[0], BC1[1]).numpy().flatten()
            u_exact = exact1(x_np)
        elif problem_id == 2:
            u_pred = hard_bc(model, x_test, BC2[0], BC2[1]).numpy().flatten()
            u_exact = exact2(x_np)
        elif problem_id == 3:
            u_pred = hard_bc(model, x_test, BC3[0], BC3[1]).numpy().flatten()
            u_exact = exact3_interp(x_np)

    return np.max(np.abs(u_pred - u_exact)), u_pred, x_np, u_exact

## PINNs as Nonlinear Collocation

### ELI5

The course taught **collocation**: pick basis functions, force the equation to hold
at sample points, solve for coefficients. A PINN does the same thing, but the
"basis functions" are inside a neural network, making the problem nonlinear.

### The Connection

| | Classical Collocation | PINN |
|---|---|---|
| Approximant | $\sum c_k \phi_k(x)$ (fixed basis) | Neural network (learned basis) |
| Unknowns | Coefficients $c_k$ | Weights $\theta$ |
| System | Linear | Nonlinear optimisation |
| Convergence | Guaranteed by theory | No guarantee |

### DOF Distinction (Critical for Fair Comparison)

- In FDM, $n$ = grid points = unknowns = sampling density. **All coupled.**
- In PINNs, architecture fixes the parameter count (~3200). $N_c$ (collocation
  points) varies independently.
- Increasing $N_c$ asks: "does more sampling help a **fixed** model?"
- This is why the comparison plot has $n$ on the x-axis for FDM and $N_c$
  for PINNs -- they measure different things.

### Spectral Bias

Networks trained with gradient descent learn **low-frequency components first**
(Rahaman et al., 2019). Smooth, polynomial-like parts of the solution are
captured quickly, but oscillatory components (like $\sin(x)$ in Prob 2)
take much longer. This is why we chose the non-polynomial version of Prob 2.

### No Convergence Theorem

- **FDM** convergence is guaranteed: $\|E^h\| \le h^2 \|A_h^{-1}\| \|\tau\|$.
- **PINN** convergence depends on the optimisation landscape. The loss can
  decrease while the true error stagnates (**loss-error decoupling**).
- FDM failure is predictable ($\kappa \cdot \epsilon_{\text{mach}} \approx 1$).
  PINN failure is not.

## Train PINNs -- Sanity Check

Train on Problem 1 first (simplest, known solution) to verify everything works.

In [45]:
# Sanity check on Problem 1
model2, hist2, t2 = train_pinn(problem_id=1, N_c=100)
err2, u_pred2, x_test2, u_exact1 = evaluate_pinn(model2, problem_id=1)
print(f'Problem 1: L-inf error = {err2:.2e}, wall time = {t2:.1f}s')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Solution
axes[0].plot(x_test2, u_exact1, 'k-', label='Exact')
axes[0].plot(x_test2, u_pred2, 'r--', label='PINN')
axes[0].set_title(f'Problem 1: PINN (error = {err2:.2e})')
axes[0].legend()
axes[0].set_xlabel('$x$')

# Loss curve
epochs, losses = zip(*hist2)
axes[1].semilogy(epochs, losses, 'b-')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Training Loss')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Problem 2: L-inf error = 6.44e-06, wall time = 94.9s


/tmp/ipykernel_1854851/824311837.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
# 5. PINN $N_c$ Sweep -- All Problems

Train PINNs at varying collocation counts to see how accuracy scales
with sampling density. Compare to FDM's $O(h^2)$ / $O(h^4)$.

**Remember the DOF distinction:** FDM couples $n$ = grid points = unknowns.
PINN architecture is fixed (~3200 params); only $N_c$ varies.
We expect the PINN error to **plateau** rather than decrease algebraically.

In [46]:
Nc_values = [10, 25, 50, 100, 250, 500, 1000]

pinn_results = {1: {}, 2: {}, 3: {}}

for pid in [1, 2, 3]:
    print(f'\n=== Problem {pid} ===')
    for Nc in Nc_values:
        model, hist, wt = train_pinn(problem_id=pid, N_c=Nc)
        err, _, _, _ = evaluate_pinn(model, problem_id=pid)
        pinn_results[pid][Nc] = {'error': err, 'wall_time': wt, 'history': hist}
        print(f'  Nc={Nc:>5}: error={err:.2e}, time={wt:.1f}s')


=== Problem 1 ===
  Nc=   10: error=1.76e-05, time=27.6s
  Nc=   25: error=1.87e-05, time=16.7s
  Nc=   50: error=2.03e-05, time=15.8s
  Nc=  100: error=2.17e-05, time=55.1s
  Nc=  250: error=2.15e-05, time=90.2s
  Nc=  500: error=2.23e-05, time=51.5s
  Nc= 1000: error=2.24e-05, time=78.0s

=== Problem 2 ===
  Nc=   10: error=8.92e-05, time=13.9s
  Nc=   25: error=9.72e-06, time=13.8s
  Nc=   50: error=7.39e-06, time=14.0s
  Nc=  100: error=6.44e-06, time=20.0s
  Nc=  250: error=6.79e-06, time=79.2s
  Nc=  500: error=6.91e-06, time=58.8s
  Nc= 1000: error=6.44e-06, time=72.7s

=== Problem 3 ===
  Nc=   10: error=5.90e-05, time=13.2s
  Nc=   25: error=6.13e-05, time=13.4s
  Nc=   50: error=6.55e-05, time=13.8s
  Nc=  100: error=6.79e-05, time=50.2s
  Nc=  250: error=6.92e-05, time=70.3s
  Nc=  500: error=6.98e-05, time=59.4s
  Nc= 1000: error=7.07e-05, time=74.2s


---
# 6. Results -- Accuracy vs Sampling Density

The **primary comparison plot** of the entire project.

Log-log axes. Three curves per problem:
- **FDM** $O(h^2)$: slope $-2$
- **Richardson** $O(h^4)$: slope $-4$
- **PINN**: expected to plateau at moderate accuracy

FDM convergence is algebraic and predictable. PINN accuracy saturates
because the model capacity is fixed (same ~3200 parameters regardless of $N_c$).

In [47]:
# FDM convergence for Problem 3
ns_fdm3 = [7, 15, 31, 63, 127, 255, 511]
errors_fdm3 = []
for n in ns_fdm3:
    # Initialize with reference solution to stay on the correct branch
    h = 1.0 / (n + 1)
    xi = np.linspace(h, 1 - h, n)
    V0 = exact3_interp(xi)
    u, x, _ = newton_fd(n, V0=V0)
    e = np.max(np.abs(exact3_interp(x[1:-1]) - u[1:-1]))
    errors_fdm3.append(e)

# Collect all results for the main comparison plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, pid, ns_fdm, e_fdm, e_rich, title in [
    (axes[0], 1, ns1, e1, r1, 'Problem 1'),
    (axes[1], 2, ns2, e2, r2, 'Problem 2'),
    (axes[2], 3, ns_fdm3, errors_fdm3, None, 'Problem 3'),
]:
    # FDM
    ax.loglog(ns_fdm, e_fdm, 'bo-', label='FDM $O(h^2)$')

    # Richardson (not for Prob 3)
    if e_rich is not None:
        rich_ns = [ns_fdm[i] for i in range(1, len(ns_fdm))]
        rich_es = [e_rich[i] for i in range(1, len(e_rich))]
        ax.loglog(rich_ns, rich_es, 'gs-', label='Richardson $O(h^4)$')

    # PINN
    pinn_nc = sorted(pinn_results[pid].keys())
    pinn_err = [pinn_results[pid][nc]['error'] for nc in pinn_nc]
    ax.loglog(pinn_nc, pinn_err, 'r^-', label='PINN')

    ax.set_xlabel('$n$ (FDM) / $N_c$ (PINN)')
    ax.set_ylabel(r'$L^\infty$ error')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('Outputs/accuracy_vs_n.pdf', bbox_inches='tight')
plt.show()
print('Saved: accuracy_vs_n.pdf')

Saved: accuracy_vs_n.pdf


/tmp/ipykernel_1854851/3912192348.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
# 7. Results -- Conditioning vs Loss Landscape

The **side-by-side failure mode comparison**.

**FDM (left):** Condition number $\kappa(A_h)$ vs $n$. Failure is
**analytically characterised**: we know exactly when $\kappa \cdot \epsilon_{\text{mach}} \approx 1$
and the solution becomes meaningless.

**PINN (right):** Training loss vs epoch for multiple $N_c$. Failure is
**empirical and unpredictable**: loss can decrease while true error stagnates
(loss-error decoupling), and there is no theory predicting when this happens.

The juxtaposition is the point: FDM's failure mode is completely described
by theory; the PINN's is not.

In [48]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: condition number
axes[0].loglog(ns_cond, kappas, 'bo-', label=r'$\kappa(A_h)$')
axes[0].loglog(ns_ref, kappa_analytic, 'r--', alpha=0.5, label=r'$4n^2/\pi^2$')
axes[0].axhline(1 / np.finfo(float).eps, color='gray', ls=':',
                label=r'$1/\epsilon_{\mathrm{mach}}$')
axes[0].set_xlabel('$n$')
axes[0].set_ylabel(r'$\kappa(A_h)$')
axes[0].set_title('FDM: Condition Number (Prob 1)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Right: PINN loss curves (Problem 1, multiple Nc)
for Nc in [10, 50, 100, 500, 1000]:
    if Nc in pinn_results[2]:
        epochs, losses = zip(*pinn_results[2][Nc]['history'])
        axes[1].semilogy(epochs, losses, label=f'$N_c={Nc}$')

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('PINN: Training Loss (Prob 1)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('Outputs/conditioning_vs_loss.pdf', bbox_inches='tight')
plt.show()
print('Saved: conditioning_vs_loss.pdf')

Saved: conditioning_vs_loss.pdf


/tmp/ipykernel_1854851/645407913.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Wall-Clock Time Comparison

FDM solves a tridiagonal system in $O(n)$ -- microseconds to milliseconds.
PINNs train a neural network for thousands of epochs -- seconds to minutes.
For 1D BVPs, FDM is orders of magnitude faster at equivalent accuracy.

In [ ]:
# FDM timing
fdm_times = {}
for pid, solver in [(1, fdm_problem1), (2, fdm_problem2)]:
    for n in [63, 255, 1023]:
        t0 = time.time()
        for _ in range(100):  # average over 100 runs
            solver(n)
        fdm_times[(pid, n)] = (time.time() - t0) / 100

# Newton FD timing for Prob 3
for n in [63, 255]:
    t0 = time.time()
    for _ in range(10):
        newton_fd(n)
    fdm_times[(3, n)] = (time.time() - t0) / 10

print('\n=== Wall-Clock Time ====')
print(f'{"Method":>20} {"Problem":>8} {"n/Nc":>8} {"Time (s)":>10} {"Error":>12}')
print('-' * 62)

# FDM entries
for (pid, n), t in sorted(fdm_times.items()):
    print(f'{"FDM":>20} {pid:>8} {n:>8} {t:>10.4e}')

# PINN entries
for pid in [1, 2, 3]:
    for Nc in [100, 500]:
        if Nc in pinn_results[pid]:
            r = pinn_results[pid][Nc]
            print(f'{"PINN":>20} {pid:>8} {Nc:>8} {r["wall_time"]:>10.2f} {r["error"]:>12.2e}')

# PINN inference timing (forward pass only, no training)
pinn_inference_times = {}
n_test = 1000
x_test = torch.linspace(0, 1, n_test + 2)[1:-1].unsqueeze(1)

for pid in [1, 2, 3]:
    # Use the Nc=100 model from the sweep (retrain to get model object)
    model, _, _ = train_pinn(problem_id=pid, N_c=100)
    bc = [BC1, BC2, BC3][pid - 1]

    # Warm up
    with torch.no_grad():
        hard_bc(model, x_test, bc[0], bc[1])

    # Time over many runs
    t0 = time.time()
    for _ in range(10000):
        with torch.no_grad():
            hard_bc(model, x_test, bc[0], bc[1])
    pinn_inference_times[pid] = (time.time() - t0) / 10000

print('\n=== PINN Inference Time (forward pass only) ===')
for pid, t in pinn_inference_times.items():
    print(f'  Problem {pid}: {t:.2e}s ({t*1e6:.1f} us)')



=== Wall-Clock Time ====
              Method  Problem     n/Nc   Time (s)        Error
--------------------------------------------------------------
                 FDM        1       63 3.0167e-04
                 FDM        1      255 2.0378e-04
                 FDM        1     1023 3.8736e-04
                 FDM        2       63 1.5056e-04
                 FDM        2      255 1.8991e-04
                 FDM        2     1023 3.4273e-04
                 FDM        3       63 8.8201e-04
                 FDM        3      255 1.4459e-03
                PINN        1      100      55.06     2.17e-05
                PINN        1      500      51.46     2.23e-05
                PINN        2      100      20.02     6.44e-06
                PINN        2      500      58.78     6.91e-06
                PINN        3      100      50.25     6.79e-05
                PINN        3      500      59.38     6.98e-05


---
# Summary

All figures saved to the `code/` directory:
- `accuracy_vs_n.pdf` -- primary comparison (FDM vs Richardson vs PINN)
- `conditioning_vs_loss.pdf` -- conditioning vs loss landscape

These can be included directly in the Beamer slides via `\includegraphics`.

In [50]:
# Save all results to files for later reading
import json as _json

results_out = {}

# --- PINN sweep results ---
results_out['pinn_results'] = {}
for pid in [1, 2, 3]:
    results_out['pinn_results'][pid] = {}
    for Nc, r in pinn_results[pid].items():
        results_out['pinn_results'][pid][Nc] = {
            'error': float(r['error']),
            'wall_time': float(r['wall_time']),
            'loss_history': [(int(e), float(l)) for e, l in r['history']],
        }

# --- PINN sanity check (Problem 1, Nc=100) ---
results_out['pinn_sanity_check'] = {
    'problem': 1,
    'Nc': 100,
    'error': float(err2),
    'wall_time': float(t2),
}

# --- PINN inference timing ---
results_out['pinn_inference_times'] = {
    int(pid): float(t) for pid, t in pinn_inference_times.items()
}

# --- PINN convergence rates (log-log slope between successive Nc) ---
results_out['pinn_convergence_rates'] = {}
for pid in [1, 2, 3]:
    ncs = sorted(pinn_results[pid].keys())
    errs = [pinn_results[pid][nc]['error'] for nc in ncs]
    rates = [None]
    for j in range(1, len(ncs)):
        r = np.log(errs[j] / errs[j-1]) / np.log(ncs[j] / ncs[j-1])
        rates.append(float(r))
    results_out['pinn_convergence_rates'][pid] = {
        'Nc': [int(x) for x in ncs],
        'rates': rates,
    }

# --- FDM convergence ---
results_out['fdm_convergence'] = {}

# Compute rates for FDM too
def compute_rates(ns, errors):
    rates = [None]
    for j in range(1, len(ns)):
        r = np.log(errors[j] / errors[j-1]) / np.log(ns[j] / ns[j-1])
        rates.append(float(r))
    return rates

results_out['fdm_convergence']['problem1'] = {
    'ns': [int(x) for x in ns1],
    'errors': [float(x) for x in e1],
    'richardson': [float(x) if x is not None else None for x in r1],
    'rates': compute_rates(ns1, e1),
}
results_out['fdm_convergence']['problem2'] = {
    'ns': [int(x) for x in ns2],
    'errors': [float(x) for x in e2],
    'richardson': [float(x) if x is not None else None for x in r2],
    'rates': compute_rates(ns2, e2),
}
results_out['fdm_convergence']['problem3'] = {
    'ns': [int(x) for x in ns_fdm3],
    'errors': [float(x) for x in errors_fdm3],
    'rates': compute_rates(ns_fdm3, errors_fdm3),
}

# --- FDM timing ---
results_out['fdm_timing'] = {
    f'prob{pid}_n{n}': float(t) for (pid, n), t in fdm_times.items()
}

# --- Condition numbers ---
results_out['condition_numbers'] = {
    'ns': [int(x) for x in ns_cond],
    'kappas': [float(x) for x in kappas],
}

# --- Nonlinear solvers ---
results_out['nonlinear_solvers'] = {
    'newton_shooting': {'alpha': float(alpha_newton), 'iterations': len(hist_newton), 'history': [float(x) for x in hist_newton]},
    'bisection': {'alpha': float(alpha_bisect), 'iterations': len(hist_bisect), 'history': [float(x) for x in hist_bisect]},
    'picard_fd': {'iterations': len(hist_picard), 'history': [float(x) for x in hist_picard]},
    'newton_fd': {'iterations': len(hist_newton_fd), 'history': [float(x) for x in hist_newton_fd]},
}

# --- Head-to-head comparison at matched n/Nc ---
results_out['head_to_head'] = {}
for pid in [1, 2, 3]:
    matched = {}
    for Nc in [100, 500]:
        if Nc in pinn_results[pid]:
            pinn_err = pinn_results[pid][Nc]['error']
            pinn_train_time = pinn_results[pid][Nc]['wall_time']
            pinn_infer_time = pinn_inference_times.get(pid, None)
            fdm_key = f'prob{pid}_n{Nc}'
            fdm_time = fdm_times.get((pid, Nc), None)
            matched[Nc] = {
                'pinn_error': float(pinn_err),
                'pinn_train_time': float(pinn_train_time),
                'pinn_inference_time': float(pinn_infer_time) if pinn_infer_time else None,
            }
    results_out['head_to_head'][pid] = matched

with open('Outputs/results.json', 'w') as f:
    _json.dump(results_out, f, indent=2)

# Save figures as PNGs too
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Re-create accuracy plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, pid, ns_fdm, e_fdm, e_rich, title in [
    (axes[0], 1, ns1, e1, r1, 'Problem 1'),
    (axes[1], 2, ns2, e2, r2, 'Problem 2'),
    (axes[2], 3, ns_fdm3, errors_fdm3, None, 'Problem 3'),
]:
    ax.loglog(ns_fdm, e_fdm, 'bo-', label='FDM $O(h^2)$')
    if e_rich is not None:
        rich_ns = [ns_fdm[i] for i in range(1, len(ns_fdm))]
        rich_es = [e_rich[i] for i in range(1, len(e_rich))]
        ax.loglog(rich_ns, rich_es, 'gs-', label='Richardson $O(h^4)$')
    pinn_nc = sorted(pinn_results[pid].keys())
    pinn_err = [pinn_results[pid][nc]['error'] for nc in pinn_nc]
    ax.loglog(pinn_nc, pinn_err, 'r^-', label='PINN')
    ax.set_xlabel('$n$ (FDM) / $N_c$ (PINN)')
    ax.set_ylabel(r'$L^\infty$ error')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Outputs/accuracy_vs_n.png', dpi=150, bbox_inches='tight')
plt.close()

# Re-create conditioning plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].loglog(ns_cond, kappas, 'bo-', label=r'$\kappa(A_h)$')
ns_ref = np.linspace(ns_cond[0], ns_cond[-1], 100)
kappa_analytic = 4 * ns_ref**2 / np.pi**2
axes[0].loglog(ns_ref, kappa_analytic, 'r--', alpha=0.5, label=r'$4n^2/\pi^2$')
axes[0].axhline(1 / np.finfo(float).eps, color='gray', ls=':', label=r'$1/\epsilon_{\mathrm{mach}}$')
axes[0].set_xlabel('$n$')
axes[0].set_ylabel(r'$\kappa(A_h)$')
axes[0].set_title('FDM: Condition Number (Prob 1)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
for Nc in [10, 50, 100, 500, 1000]:
    if Nc in pinn_results[1]:
        epochs, losses = zip(*pinn_results[1][Nc]['history'])
        axes[1].semilogy(epochs, losses, label=f'$N_c={Nc}$')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('PINN: Training Loss (Prob 1)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Outputs/conditioning_vs_loss.png', dpi=150, bbox_inches='tight')
plt.close()

# Solver convergence plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].semilogy(hist_newton, 'ro-', label='Newton shooting')
axes[0].semilogy(hist_bisect, 'bs-', label='Bisection')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel(r'$|\Phi(\alpha)|$')
axes[0].set_title('Shooting Methods')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].semilogy(hist_picard, 'bs-', label='Picard FD')
axes[1].semilogy(hist_newton_fd, 'ro-', label='Newton FD')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel(r'$\|V_{k+1} - V_k\| / \sqrt{n}$')
axes[1].set_title('FD Iterative Methods')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Outputs/solvers_convergence.png', dpi=150, bbox_inches='tight')
plt.close()

print('Saved: results.json, accuracy_vs_n.png, conditioning_vs_loss.png, solvers_convergence.png')

Saved: results.json, accuracy_vs_n.png, conditioning_vs_loss.png, solvers_convergence.png
